# 02 LDA — 토픽 모델링 + K 선택(c_npmi·perplexity·시드 안정성 3축)

01의 `분석코퍼스_*.csv`에서 `in_universe`만 골라 LDA로 그 주 핵심 의제를 뽑는 노트북

- BoW = `CountVectorizer(analyzer=str.split)` — 입력이 공백조인 토큰이라 split 기반(기본 token_pattern은 1글자 한자 버림)
- 불용어는 **벡터화 전 토큰열에서 직접 제거** + 단일 라틴(`^[A-Za-z]$`) 제거, 한자 1글자는 보존
- K 탐색 6~15 — 단일 c_v 금지: **c_npmi 주 + c_v 보조 + held-out perplexity 보조 + 시드 10회 안정성**
- ★ **누수 차단** — 분할을 먼저 하고 vocab을 train으로만 학습, coherence·perplexity·안정성은 train 모델로. 최종 theta/토픽만 전체 코퍼스 재적합
- 의제 비중 주분석은 **soft assignment(theta 가중)** — 문서 안 버림. hard 라벨(`p1≥0.35 & margin≥0.10`)은 03 하위분석용


## K 선택 규칙 (사전 고정)
1. **안정성 게이트(hard)** — 시드 간 beta 헝가리안 정렬 후 top-20 Jaccard/beta-cosine, 토픽이 평균 Jaccard<0.5 **그리고** beta-cosine<0.7이면 불안정. K 내 불안정 토픽 1개 초과면 그 K 배제. (기준 시드는 시드쌍 정렬점수 최고 = medoid)
2. 통과 후보 중 **c_npmi plateau** — c_npmi 최고값의 1 SD 이내(1SD=각 K 시드 반복 간 SD)
3. 그 안에서 **held-out perplexity 안 악화(+2% 이내) + 중복 토픽(top-10 Jaccard≥0.5) ≤2개**인 **가장 작은 K**. 규칙 만족 후보 없으면 plateau 최소 K로 폴백하되 **경고 + 수동 검토**
4. 상한 15에서도 c_npmi가 계속 오르면 상한 재확장


In [ ]:
# Colab에서 실행할 때만 아래 주석 해제
# from google.colab import drive
# drive.mount('/content/drive')
# !pip install gensim


In [ ]:
from pathlib import Path
import os, re, unicodedata
import numpy as np, pandas as pd

try:
    PROJECT_DIR = Path('/content/drive/MyDrive/Text-data-Analysis_26-Spring')
    if not PROJECT_DIR.exists():
        raise FileNotFoundError
except Exception:
    PROJECT_DIR = Path('/home/carol/Text-data-Analysis_26-Spring')
os.chdir(PROJECT_DIR)
RESULT_DIR = PROJECT_DIR / 'result'

def normalize_name(p): return unicodedata.normalize('NFC', p.name)
cands = sorted(p for p in RESULT_DIR.iterdir()
               if p.is_file() and re.match(r'^분석코퍼스_언론사_\d{6}_\d{6}\.csv$', normalize_name(p)))
if not cands:
    raise FileNotFoundError('01 산출 분석코퍼스를 먼저 만들 것(01 노트북 실행)')
CORPUS_PATH = cands[-1]
if len(cands) > 1:
    print('⚠ result/에 여러 기간 분석코퍼스 존재 — 최신', normalize_name(CORPUS_PATH), '사용. 의도 확인')
PERIOD = re.search(r'(\d{6}_\d{6})', normalize_name(CORPUS_PATH)).group(1)

corpus = pd.read_csv(CORPUS_PATH, encoding='utf-8-sig')
univ = corpus[corpus['in_universe'] == True].copy().reset_index(drop=True)   # ★ in_universe만
print('우주 문서:', len(univ), '/ 그룹:', univ['media_group'].value_counts().to_dict())


In [ ]:
# --- 불용어(벡터화 전 토큰열에서 직접 제거) + 단일 라틴 제거 ---
# 실측 고DF 추상어(우주 기준 DF로 재점검해 보강). 잠시만요·관련은 DF=0이라 무의미
DOMAIN_STOP = {
    '가능','시간','확인','문제','발생','내용','생각','정도','경우','이번','지난','최근',
    '관련','대해','통해','위해','상황','모습','부분','이후','현재','전망','계획','예정',
}
SINGLE_LATIN = re.compile(r'^[A-Za-z]$')   # Q·W·H 등 단일 알파벳 노이즈(일부 min_df 통과) — 한자 1글자는 보존

def clean_tokens(s):
    out = []
    for t in str(s).split():
        if t in DOMAIN_STOP:        continue
        if SINGLE_LATIN.match(t):   continue
        out.append(t)
    return out

univ['toks'] = univ['tokens'].map(clean_tokens)

# zero-token audit — 불용어 제거로 토큰 0개가 된 문서는 LDA서 제외(수 리포트)
n_before = len(univ)
empty_mask = univ['toks'].map(len) == 0
print('불용어 제거 후 zero-token 문서:', int(empty_mask.sum()), '→ LDA서 제외')
univ = univ[~empty_mask].reset_index(drop=True)
print('LDA 입력 문서:', len(univ), f'(우주 {n_before}에서 {n_before-len(univ)} 제외)')
docs_joined = univ['toks'].map(lambda x: ' '.join(x)).tolist()   # CountVectorizer(analyzer=str.split) 입력


In [ ]:
# --- ★ 분할 먼저(누수 차단) → vocab은 train으로만 학습 ---
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from gensim.corpora import Dictionary
MIN_DF = 5

# train/test 80/20 — media_group×date 층화(대표성). 1건 층은 분할 불가라 통합
strata = (univ['media_group'].astype(str) + '|' + univ['date'].astype(str))
vc = strata.value_counts()
strat_arg = strata.where(strata.map(vc) >= 2, other='__rare__')
idx_tr, idx_te = train_test_split(np.arange(len(univ)), test_size=0.2, random_state=0, stratify=strat_arg)
docs_tr = [docs_joined[i] for i in idx_tr]
docs_te = [docs_joined[i] for i in idx_te]

# ★ vocab은 train으로만 fit — test 문서가 min_df 임계에 영향 못 주게(누수 차단). K탐색 진단은 전부 train 어휘/모델 기준
vec_tr = CountVectorizer(analyzer=str.split, min_df=MIN_DF).fit(docs_tr)
vocab_tr = vec_tr.get_feature_names_out(); vset_tr = set(vocab_tr)
X_tr = vec_tr.transform(docs_tr); X_te = vec_tr.transform(docs_te)   # test는 train 어휘로 transform(OOV 자동 제거)
print('train vocab:', len(vocab_tr), '/ X_tr', X_tr.shape, '/ X_te', X_te.shape)
print('train 어휘 한자 잔존:', [w for w in ['美','韓','中','與','尹','靑','北','野'] if w in vset_tr])

# coherence는 평가 대상(train LDA)과 일치 — train 텍스트·train 어휘로
texts_tr = [[t for t in univ['toks'].iloc[i] if t in vset_tr] for i in idx_tr]
dict_tr = Dictionary(texts_tr)


In [ ]:
# --- 헬퍼: LDA 적합 / top words / coherence / 안정성(헝가리안) ---
from sklearn.decomposition import LatentDirichletAllocation
from gensim.models import CoherenceModel
from scipy.optimize import linear_sum_assignment

def fit_lda(k, seed, Xfit):
    return LatentDirichletAllocation(n_components=k, random_state=seed, learning_method='batch',
                                     max_iter=50, n_jobs=-1).fit(Xfit)   # max_iter 50 — batch 수렴 여유

def top_words(lda, vocab, topn=20):
    return [[vocab[j] for j in comp.argsort()[::-1][:topn]] for comp in lda.components_]

def coherence(topic_words, measure, texts, dictionary):
    return CoherenceModel(topics=topic_words, texts=texts, dictionary=dictionary, coherence=measure, topn=20).get_coherence()

def beta_norm(lda):
    return lda.components_ / lda.components_.sum(axis=1, keepdims=True)

def pair_align(beta_ref, tw_ref, beta_o, tw_o, topn=20):
    # 헝가리안: beta 코사인 최대 매칭 → 매칭쌍별 top-20 Jaccard, beta-cosine(참조 토픽 순서)
    A = beta_ref / np.linalg.norm(beta_ref, axis=1, keepdims=True)
    B = beta_o  / np.linalg.norm(beta_o,  axis=1, keepdims=True)
    cos = A @ B.T
    r, c = linear_sum_assignment(-cos)
    jac = np.array([len(set(tw_ref[i][:topn]) & set(tw_o[j][:topn])) /
                    len(set(tw_ref[i][:topn]) | set(tw_o[j][:topn])) for i, j in zip(r, c)])
    bcos = np.array([float(cos[i, j]) for i, j in zip(r, c)])
    return jac, bcos


In [ ]:
# --- K 탐색 6~15 × 시드 10회 (전부 train 모델) ---
K_RANGE = list(range(6, 16))
N_SEEDS = 10              # plan 고정값 — 무거우면 낮춰 1차 탐색 후 확정시 10으로
DUP_TOPN, DUP_JAC = 10, 0.5

rows = []; stab_long = []
for k in K_RANGE:
    ldas  = [fit_lda(k, s, X_tr) for s in range(N_SEEDS)]
    tws   = [top_words(m, vocab_tr) for m in ldas]
    betas = [beta_norm(m) for m in ldas]
    cnpmi = [coherence(tw, 'c_npmi', texts_tr, dict_tr) for tw in tws]
    cv0   = coherence(tws[0], 'c_v', texts_tr, dict_tr)              # c_v는 느려 seed0만(보조·단일시드라 노이즈 있음)
    perp  = [ldas[s].perplexity(X_te) for s in range(N_SEEDS)]       # ★ 진짜 held-out(train 어휘)
    # 시드쌍 정렬 캐시 + medoid 기준 시드(seed0 치우침 방지)
    cache = {}; align_score = np.zeros(N_SEEDS)
    for a in range(N_SEEDS):
        for b in range(N_SEEDS):
            if a == b: continue
            jac, bcos = pair_align(betas[a], tws[a], betas[b], tws[b]); cache[(a,b)] = (jac, bcos)
            align_score[a] += bcos.mean()
    ref = int(align_score.argmax())
    jac_acc = np.zeros(k); bcos_acc = np.zeros(k); cnt = 0
    for b in range(N_SEEDS):
        if b == ref: continue
        jac, bcos = cache[(ref, b)]; jac_acc += jac; bcos_acc += bcos; cnt += 1
    jac_pt, bcos_pt = jac_acc/cnt, bcos_acc/cnt
    unstable = int(np.sum((jac_pt < 0.5) & (bcos_pt < 0.7)))        # 게이트: 둘 다 미달이면 불안정
    for t in range(k):
        stab_long.append(dict(K=k, topic=t, jaccard=round(float(jac_pt[t]),4),
                              betacos=round(float(bcos_pt[t]),4),
                              unstable=bool((jac_pt[t]<0.5) and (bcos_pt[t]<0.7))))
    dup = 0
    for a in range(k):
        for b in range(a+1, k):
            s1, s2 = set(tws[ref][a][:DUP_TOPN]), set(tws[ref][b][:DUP_TOPN])
            if len(s1 & s2)/len(s1 | s2) >= DUP_JAC: dup += 1
    rows.append(dict(K=k, c_npmi=np.mean(cnpmi), c_npmi_sd=np.std(cnpmi),
                     c_v_seed0=cv0, perplexity=np.mean(perp), unstable=unstable, dup_topics=dup, ref_seed=ref))
    print(f'K={k:2d}  c_npmi={np.mean(cnpmi):.4f}±{np.std(cnpmi):.4f}  c_v0={cv0:.4f}  perp={np.mean(perp):.1f}  unstable={unstable}  dup={dup}  ref={ref}')
metrics = pd.DataFrame(rows).set_index('K')
stab_df = pd.DataFrame(stab_long)


In [ ]:
# --- K 선택 규칙 적용 ---
gate = metrics[metrics['unstable'] <= 1]     # ① 안정성 게이트
print('안정성 게이트 통과 K:', list(gate.index))
if len(gate) == 0:
    print('⚠ 게이트 통과 K 없음 — 게이트 완화/전처리 재점검 필요. 전체 K로 폴백'); gate = metrics

best_k = gate['c_npmi'].idxmax()             # ② c_npmi plateau(최고값 1SD 이내)
thr = gate.loc[best_k,'c_npmi'] - metrics.loc[best_k,'c_npmi_sd']
plateau = gate[gate['c_npmi'] >= thr]
print(f'c_npmi 최고 K={best_k} (={gate.loc[best_k,"c_npmi"]:.4f}), plateau 임계 {thr:.4f} → 후보 {list(plateau.index)}')

perp_min = plateau['perplexity'].min()        # ③ plateau 후보 내 최저 대비 +2% 이내 + 중복토픽 ≤2 인 가장 작은 K(게이트 탈락 K는 기준서 제외)
sel = plateau[(plateau['perplexity'] <= perp_min*1.02) & (plateau['dup_topics'] <= 2)]
if len(sel):
    CHOSEN_K = int(sel.index.min())
else:
    CHOSEN_K = int(plateau.index.min())
    print(f'⚠ perplexity(+2%)·중복토픽(≤2) 만족 후보 없음 — plateau 최소 K={CHOSEN_K}로 폴백. 사람 검토·수동 덮어쓰기 권장')
print('▶ CHOSEN_K =', CHOSEN_K)
if metrics['c_npmi'].idxmax() == max(K_RANGE):
    print('⚠ c_npmi가 상한 15에서 최고 — K_RANGE 상한 재확장 검토')
# CHOSEN_K = 8   # ← 사람 검토 후 덮어쓰려면 주석 해제


In [ ]:
# --- 확정 K를 전체 코퍼스에 재적합(별도 vocab) → theta/dominant/하위분석 표본 ---
vec_full = CountVectorizer(analyzer=str.split, min_df=MIN_DF).fit(docs_joined)   # ★ 최종은 전체 코퍼스 vocab
vocab_full = vec_full.get_feature_names_out()
X_full = vec_full.transform(docs_joined)
print('full vocab:', len(vocab_full), '/ X_full', X_full.shape)

final = fit_lda(CHOSEN_K, 0, X_full)
theta = final.transform(X_full); theta = theta / theta.sum(axis=1, keepdims=True)
K = CHOSEN_K

order = np.argsort(-theta, axis=1)
p1 = theta[np.arange(len(theta)), order[:,0]]
p2 = theta[np.arange(len(theta)), order[:,1]]
dom = order[:,0]
ent = -(theta*np.log(theta+1e-12)).sum(axis=1) / np.log(K)    # 정규화 엔트로피(저=집중) — 민감도용 컬럼, 라벨엔 미적용

univ['dominant_topic'] = dom
univ['p1'] = p1; univ['p2'] = p2; univ['margin'] = p1 - p2; univ['entropy_norm'] = ent
# ★ hard 고순도 표본(하위분석용): p1≥0.35 AND margin≥0.10 만으로 결정(엔트로피는 라벨에 미적용, 민감도 전용)
univ['high_purity'] = (univ['p1'] >= 0.35) & (univ['margin'] >= 0.10)
for j in range(K):
    univ[f'theta_{j}'] = theta[:, j]

unc_rate = (1 - univ.groupby('media_group')['high_purity'].mean()).mul(100).round(1)
print('그룹별 uncertain(hard 비선정) 비율 %:'); print(unc_rate)
print('hard 고순도 표본 N:', int(univ['high_purity'].sum()), '/', len(univ), '(주분석 soft theta는 문서 안 버림)')


In [ ]:
# --- 산출 저장 ---
# 최종 토픽 품질 c_npmi(full 텍스트·full 어휘, 보고용 — 평가 모델과 일치)
vset_full = set(vocab_full)
texts_full = [[t for t in toks if t in vset_full] for toks in univ['toks']]
dict_full = Dictionary(texts_full)
tw_final = top_words(final, vocab_full, topn=20)
final_cnpmi = coherence(tw_final, 'c_npmi', texts_full, dict_full)
print('최종 K', K, '/ topic c_npmi(full):', round(final_cnpmi, 4))

topic_tbl = pd.DataFrame({'topic': range(K),
                          'top_words': [' '.join(w) for w in tw_final],
                          'topic_mass': [theta[:,j].mean() for j in range(K)]})
topic_tbl = topic_tbl.sort_values('topic_mass', ascending=False)
topic_tbl.to_csv(RESULT_DIR / f'토픽_대표단어_언론사_{PERIOD}.csv', index=False, encoding='utf-8-sig')
print('핵심 의제(topic mass 상위):'); print(topic_tbl.head(6).to_string(index=False))

keep = ['article_id','media_group','press','date','article_category','title_cleaned','n_tokens',
        'dominant_topic','p1','p2','margin','entropy_norm','high_purity'] + [f'theta_{j}' for j in range(K)]
univ[keep].to_csv(RESULT_DIR / f'문서토픽분포_언론사_{PERIOD}.csv', index=False, encoding='utf-8-sig')
metrics.to_csv(RESULT_DIR / f'K선택지표_언론사_{PERIOD}.csv', encoding='utf-8-sig')
stab_df.to_csv(RESULT_DIR / f'K안정성_per_topic_언론사_{PERIOD}.csv', index=False, encoding='utf-8-sig')
print('저장: 토픽_대표단어 / 문서토픽분포 / K선택지표 / K안정성_per_topic')


## 검증 체크리스트
- ★ 누수 차단 — vocab은 train으로만 fit, perplexity는 train 어휘로 transform한 test에서, coherence도 train 텍스트/어휘. 최종 theta/토픽만 전체 코퍼스 재적합
- BoW에 1글자 한자(美·韓 등) 잔존, 단일 라틴 제거, zero-token 제외 수 리포트
- K별 (c_npmi±SD·c_v·perplexity·unstable·dup) 표 + 선택 규칙(게이트→plateau→perplexity/중복→최소 K)대로 CHOSEN_K, 폴백 시 경고
- per-topic 안정성 표(K안정성_per_topic) 저장 — 게이트 근거 감사 가능
- 토픽 대표단어가 실이슈(호르무즈 한국선박·6·3 지방선거·이란/미국 등)와 맞는지 사람 확인
- 의제 비중 주분석은 theta(soft) — 문서 안 버림. hard 고순도는 하위분석용, 그룹별 uncertain율 같이 봄
